Name: Ivy Cheng

Github Username: ivychengusc

USC ID: 6931463618

In [13]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1(a)

In [14]:
data_path = '../data/AReM'

activities = [
    'bending1',
    'bending2',
    'cycling',
    'lying',
    'sitting',
    'standing',
    'walking'
]

for activity in activities:
    folder_path = os.path.join(data_path, activity)
    print(activity, len([
        file for file in os.listdir(folder_path)
        if file.endswith('.csv')
    ]))

bending1 7
bending2 6
cycling 15
lying 15
sitting 15
standing 15
walking 15


## 1(b)

In [15]:
train_files = []
test_files = []

for activity in activities:
    
    folder_path = os.path.join(
        data_path,
        activity
    )
    
    files = [
        file for file in os.listdir(folder_path)
        if file.startswith('dataset')
        and file.endswith('.csv')
    ]
    
    files = sorted(
        files,
        key=lambda x: int(
            x.replace('dataset', '')
             .replace('.csv', '')
        )
    )
    
    for file in files:
        
        dataset_number = int(
            file.replace('dataset', '')
                .replace('.csv', '')
        )
        
        file_path = os.path.join(
            folder_path,
            file
        )
        
        if activity in ['bending1', 'bending2']:
            
            if dataset_number <= 2:
                test_files.append(
                    (activity, file_path)
                )
            else:
                train_files.append(
                    (activity, file_path)
                )
        
        else:
            
            if dataset_number <= 3:
                test_files.append(
                    (activity, file_path)
                )
            else:
                train_files.append(
                    (activity, file_path)
                )

print("Training instances:", len(train_files))
print("Test instances:", len(test_files))

Training instances: 69
Test instances: 19


## 1(c)(i)
Commonly used time-domain features for time series classification include minimum, maximum, mean, median, standard deviation, variance, range, first quartile, third quartile, interquartile range, root mean square, skewness, kurtosis, zero-crossing rate, autocorrelation.

## 1(c)(ii)

In [16]:
time_series_names = [
    'avg_rss12',
    'var_rss12',
    'avg_rss13',
    'var_rss13',
    'avg_rss23',
    'var_rss23'
]

In [17]:
def read_activity_file(file_path):
    
    with open(file_path, 'r') as file:
        
        for line in file:
            
            if (
                line.strip()
                and not line.startswith('#')
            ):
                
                if ',' in line:
                    separator = ','
                else:
                    separator = r'\s+'
                
                break
    
    data = pd.read_csv(
        file_path,
        comment='#',
        header=None,
        sep=separator,
        usecols=range(7),
        engine='python'
    )
    
    data.columns = [
        'time',
        'avg_rss12',
        'var_rss12',
        'avg_rss13',
        'var_rss13',
        'avg_rss23',
        'var_rss23'
    ]
    
    return data

In [18]:
all_files = []

for activity in activities:
    
    folder_path = os.path.join(
        data_path,
        activity
    )
    
    files = [
        file for file in os.listdir(folder_path)
        if file.startswith('dataset')
        and file.endswith('.csv')
    ]
    
    files = sorted(
        files,
        key=lambda x: int(
            x.replace('dataset', '')
             .replace('.csv', '')
        )
    )
    
    for file in files:
        
        all_files.append(
            os.path.join(
                folder_path,
                file
            )
        )

print("Number of instances:", len(all_files))

Number of instances: 88


In [19]:
feature_rows = []

for instance_number, file_path in enumerate(
    all_files,
    start=1
):
    
    data = read_activity_file(
        file_path
    )
    
    row = {
        'Instance': instance_number
    }
    
    for i, column in enumerate(
        time_series_names,
        start=1
    ):
        
        series = data[column]
        
        row['min' + str(i)] = (
            series.min()
        )
        
        row['max' + str(i)] = (
            series.max()
        )
        
        row['mean' + str(i)] = (
            series.mean()
        )
        
        row['median' + str(i)] = (
            series.median()
        )
        
        row['std' + str(i)] = (
            series.std()
        )
        
        row['1st_quart' + str(i)] = (
            series.quantile(0.25)
        )
        
        row['3rd_quart' + str(i)] = (
            series.quantile(0.75)
        )
    
    feature_rows.append(row)

In [20]:
feature_df = pd.DataFrame(
    feature_rows
)

feature_df.head()

,Instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [21]:
print(feature_df.shape)

(88, 43)


In [22]:
print(
    "Missing values:",
    feature_df.isna().sum().sum()
)

Missing values: 0


For each of the six time series in every instance, I extracted the following seven time-domain features: minimum, maximum, mean, median, standard deviation, first quartile, third quartile.

Therefore, each instance has

$$
6 \times 7 = 42
$$

time-domain features.

Since the dataset contains 88 instances, the resulting feature dataset contains 88 rows and 42 extracted features. An additional Instance column is included for identification, so the DataFrame has 43 columns in total.

## 1(c)(iii)

In [23]:
feature_columns = feature_df.columns.drop('Instance')

np.random.seed(42)

bootstrap_results = []

for feature in feature_columns:
    
    values = feature_df[feature].to_numpy()
    
    estimated_std = np.std(
        values,
        ddof=1
    )
    
    bootstrap_std = []
    
    for i in range(1000):
        
        bootstrap_sample = np.random.choice(
            values,
            size=len(values),
            replace=True
        )
        
        bootstrap_std.append(
            np.std(
                bootstrap_sample,
                ddof=1
            )
        )
    
    lower_bound = np.percentile(
        bootstrap_std,
        5
    )
    
    upper_bound = np.percentile(
        bootstrap_std,
        95
    )
    
    bootstrap_results.append([
        feature,
        estimated_std,
        lower_bound,
        upper_bound
    ])

In [24]:
std_ci_table = pd.DataFrame(
    bootstrap_results,
    columns=[
        'Feature',
        'Standard Deviation',
        '90% CI Lower',
        '90% CI Upper'
    ]
)

std_ci_table.round(3)

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.570,8.267,10.693
1,max1,4.394,3.370,5.293
2,mean1,5.336,4.720,5.902
3,median1,5.440,4.763,6.018
4,std1,1.772,1.571,1.953
5,1st_quart1,6.154,5.546,6.660
6,3rd_quart1,5.139,4.388,5.841
7,min2,0.000,0.000,0.000
8,max2,5.063,4.627,5.404
9,mean2,1.574,1.402,1.714


The table above shows the estimated standard deviation and the 90% bootstrap confidence interval for each of the 42 time-domain features.

The sample standard deviation is calculated across the 88 activity instances. For each feature, 1,000 bootstrap samples are generated by sampling the 88 observations with replacement.

A 90% percentile bootstrap confidence interval is obtained using the 5th and 95th percentiles of the bootstrap standard deviations.

## 1(c)(iv)
I would select mean, standard deviation, and maximum as the three most important time-domain features.

Mean represents the overall level of a time series and can help distinguish activities that produce different average sensor measurements. Standard deviation measures the amount of variation in a time series. Activities involving more movement are likely to have greater variability than relatively stationary activities. Maximum captures the peak value of a time series and may help identify activities that produce large or sudden sensor readings.

Together, these three features describe the central tendency, variability, and extreme behavior of each time series, which makes them useful for distinguishing different types of activities.

## 2(a)

The cubic regression will have training RSS less than or equal to the training RSS of the linear regression.

The linear model is

$$
Y = \beta_0 + \beta_1 X + \epsilon,
$$

while the cubic model is

$$
Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \epsilon.
$$

The linear model is a special case of the cubic model when

$$
\beta_2 = \beta_3 = 0.
$$

Therefore, the cubic model is more flexible and can fit the training data at least as well as the linear model. Even if the true relationship is linear, the extra terms in the cubic model can fit some of the noise in the training data.

Thus,

$$
RSS_{\text{train,cubic}} \leq RSS_{\text{train,linear}}.
$$


## 2(b)

For test RSS, we would generally expect the linear regression to have lower test RSS.

Since the true relationship between $X$ and $Y$ is linear, the linear model has the correct functional form. The cubic model includes unnecessary $X^2$ and $X^3$ terms, which increase the flexibility and variance of the model.

The cubic model may fit the training data better, but it may also fit some of the noise and therefore perform worse on new observations.

Thus, we generally expect

$$
RSS_{\text{test,linear}} < RSS_{\text{test,cubic}}.
$$


## 2(c)

If the true relationship between $X$ and $Y$ is nonlinear, the cubic regression will still have training RSS less than or equal to the training RSS of the linear regression.

The cubic model contains the linear model as a special case and has additional $X^2$ and $X^3$ terms. Therefore, it is more flexible and can fit the training data at least as well as the linear model.

Thus,

$$
RSS_{\text{train,cubic}} \leq RSS_{\text{train,linear}}.
$$

Since the true relationship is nonlinear, we would generally expect the cubic model to have lower training RSS.


## 2(d)

For test RSS, there is not enough information to determine which model will perform better.

The true relationship is nonlinear, so the cubic model may perform better if the nonlinear relationship is well represented by a cubic function. In that case, the cubic model would have lower bias than the linear model.

However, we do not know how nonlinear the true relationship is or whether a cubic function is a good approximation. The additional flexibility of the cubic model also increases variance and may lead to overfitting.

Therefore, either model could have lower test RSS depending on the true relationship and the amount of noise in the data.